In [238]:
from caat import CAAT

def filter_sne(not_na_cols=[], **kwargs):

    caat = CAAT()
    filtered_sne = caat.get_list_of_sne(**kwargs)

    for col in not_na_cols:
        filtered_sne = filtered_sne.dropna(subset=[col])
    
    return filtered_sne

filter_sne(not_na_cols=["Filtmax"], Type="SESNe", Subtype="SNIb")


,Name,Type,Subtype,Redshift,RA,Dec,Tmax,Magmax,Filtmax
1,SN2004dk,SESNe,SNIb,0.005200,245.453880,-2.271470,53235.767258,15.810637,B
4,SN2022crv,SESNe,SNIb,0.008091,148.607958,-25.703100,59654.187835,15.447415,o
5,SN2007C,SESNe,SNIb,0.005611,197.205420,-6.783610,54118.300000,16.746993,g
6,SN2021uhk,SESNe,SNIb,0.035000,351.556648,1.145571,59437.254075,18.807625,g
7,SN2019yvr,SESNe,SNIb,0.005000,191.283897,-0.459120,58854.109192,16.062443,g
8,SN2023dtc,SESNe,SNIb,0.006000,128.323125,-22.962519,60036.148287,17.515727,g
9,SN2019dge,SESNe,SNIb,0.021300,264.194759,50.547811,58583.727828,18.514179,g
10,SN2019dgz,SESNe,SNIb,0.036400,184.019172,29.851698,58591.268081,19.200746,g
12,SN2009jf,SESNe,SNIb,0.005200,185.728875,15.823600,55121.245057,15.070219,V
14,SN2018beh,SESNe,SNIb,0.051000,142.845973,17.807700,58250.910498,17.771295,V


In [307]:
import json
import os
import requests

import pandas as pd
import numpy as np

from caat import SN
from caat.utils import WLE


"""
Make full sample LaTeX table

Prints LaTeX code to generate a table describing
every object in our sample.
Contains columns for name, type and subtype, redshift, peak information, 
number of detections, filters with detections, and phase range of detections.
"""

type_dict = {
    "SESNe": [
        "SNIIb",
        "SNIb",
        "SNIc"
    ],
    "SNII": [
        "SNII",
        "SNII-pec",
        "SNIIP",
    ],
    "SNIIn": ["SNIIn"],
    "FBOT": [
        "SNIbn",
        "SNIcn",
    ],
    "SLSN-I": ["SLSN-I"],
    "SLSN-II": ["SLSN-II"],
    "Other": [
        "SNIa",
        "SNIa-91bg-like",
        "SNIa-91T-like",
        "SNIa-CSM",
        "SNIa-pec",
        "SNIax",
        "TDE",
    ]
}

def get_opensn_data_refs(sn: SN):
    """
    Get references to the data on OSC that we aggregate for a given SN object
    """
    OPENSN_BASE_URL = "https://api.astrocats.space/"
    OPENSN_PHOT_URL = "/photometry/magnitude+e_magnitude+band+time+telescope+source"
    
    dirfiles = os.listdir(
        os.path.join(sn.base_path, sn.classification, sn.subtype, sn.name)
    )
    filts = None
    for file in dirfiles:
        if "opensn" in file:
            with open(
                os.path.join(
                    sn.base_path,
                    sn.classification,
                    sn.subtype,
                    sn.name,
                    f"{sn.name}_opensn.json"
                ),
                "r",
            ) as f:
                osc_data = json.load(f)
            filts = list(osc_data.keys())
            break

    if filts is None:
        return []
    
    response = json.loads(requests.get(OPENSN_BASE_URL + sn.name + OPENSN_PHOT_URL).text)
    if sn.name not in response.keys(): # Not found
        return []
    
    opensn_refs = []
    for phot in response[sn.name]['photometry']:
        if phot[2] in filts:
            # opensn_refs.setdefault(phot[2], []).extend([phot[5]])
            opensn_refs.append(phot[5])

    return list(set(opensn_refs))


# Print the header line of the table
print("Name & Type & Subtype & Redshift & Peak Date & Peak Magnitude & Peak Filter & # Detections & Filters with Detections & Phase of First Detection & Phase of Last Detection & Data Sources & References\\\\")

# Store a list of references
refs = []

with open("sample_table.txt", "a+") as f:
    for typ, subtyp_list in type_dict.items():
        for subtyp in subtyp_list:
            sn_df = filter_sne(not_na_cols=["Filtmax", "Redshift"], Type=typ, Subtype=subtyp)
            sn_df = sn_df.sort_values("Name", ascending=False)
            for _, row in sn_df.iterrows():
                name = row["Name"]
                try:
                    sn = SN(name=name)
                except:
                    sn = SN(name="AT"+name[2:])

                # Get number of detections
                data_cube_filename = os.path.join(
                    sn.base_path,
                    sn.classification,
                    sn.subtype,
                    sn.name,
                    sn.name + "_datacube_mangled.csv"
                )
                if os.path.exists(data_cube_filename):
                    cube = pd.read_csv(data_cube_filename)
                    dets = cube[cube["Nondetection"] == False]  # noqa: E712
                    number_of_dets = len(dets)

                    # Get filters with detections
                    filters_with_dets = list(set(dets["Filter"]))
                    filter_list = sorted(filters_with_dets, key=lambda x: WLE.get(x, len(WLE)))
                    filters = ",".join(filt for filt in filter_list)
                    if len(filters.replace(",", "")) > 12:
                        first_filts = []
                        count = 0
                        for filt in filters.split(','):
                            if count < 11:
                                count += len(filt)
                                first_filts.append(filt)
                        second_filts = [filt for filt in filters.split(',') if filt not in first_filts]
                        filts_1 = ",".join(filt for filt in sorted(first_filts, key=lambda x: WLE.get(x, len(WLE))))
                        filts_2 = ",".join(filt for filt in sorted(second_filts, key=lambda x: WLE.get(x, len(WLE))))
                        filters = f"\shortstack{{{filts_1} \\\\ {filts_2}}}"

                    # Get phase of first and last detections
                    phase_of_first_det = min(dets["Phase"])
                    phase_of_last_det = max(dets["Phase"])

                    # Get data sources
                    data_sources = []
                    dirfiles = os.listdir(
                        os.path.join(sn.base_path, sn.classification, sn.subtype, sn.name)
                    )
                    for file in dirfiles:
                        if "atlas" in file:
                            data_sources.append("ATLAS")
                        if "asassn" in file:
                            data_sources.append("ASAS-SN")
                        if "cfa" in file:
                            data_sources.append("CfA")
                        if "opensn" in file:
                            data_sources.append("OSC")
                        if "uvot" in file:
                            data_sources.append("Swift")
                        if "ztf" in file:
                            data_sources.append("ZTF")
                    data_source = ", ".join(source for source in sorted(list(set(data_sources))))
                    if len(data_source.replace(", ", "")) > 12:
                        first_srcs = []
                        count = 0
                        for src in data_source.replace(" ", "").split(','):
                            if count < 11:
                                count += len(src)
                                first_srcs.append(src)
                        second_srcs = [src for src in data_source.replace(" ", "").split(',') if src not in first_srcs]
                        if len(second_srcs) > 0:
                            data_source_1 = ", ".join(source for source in sorted(list(set(first_srcs))))
                            data_source_2 = ", ".join(source for source in sorted(list(second_srcs)))
                            data_source = f"\shortstack{{{data_source_1} \\\\ {data_source_2}}}"

                    # Get refs from OSC
                    
                    opensn_refs = get_opensn_data_refs(sn)
                    ref_str = ''
                    if len(opensn_refs) > 0:
                        current_refs = [ref for refstring in opensn_refs for ref in refstring.split(',') if ref.startswith('20') or ref.startswith('19')]
                        if len(current_refs) > 3:
                            ref_str = "\shortstack{"
                        current_count = 0
                        for ref in current_refs:
                            try:
                                ref_ind = refs.index(ref) + 1
                            except ValueError:                            
                                refs.append(ref)
                                ref_ind = len(refs)

                            if current_count == 3:
                                ref_str += "\\\\ "
                                current_count = 0

                            ref_str += f"{ref_ind} "
                            current_count += 1
                    
                        if len(current_refs) > 3:
                            ref_str += "}"

                    # f.write(f"{name} & {typ} & {subtyp} & {str(row['Redshift'])[:6]} & {round(row['Tmax'], 1)} & {round(row['Magmax'], 2)} & {row['Filtmax']} & {number_of_dets} & {filters} & {round(phase_of_first_det, 1)} & {round(phase_of_last_det, 1)} & {data_source} & {ref_str}\\\\ \n")
                    # f.write("\hline\n")

                #else:
                    #print("Missing datacube for ", sn.name)

    for i, ref in enumerate(refs):
        f.write(f"[{i+1}]: \citet{{{ref}}}; ")

Name & Type & Subtype & Redshift & Peak Date & Peak Magnitude & Peak Filter & # Detections & Filters with Detections & Phase of First Detection & Phase of Last Detection & Data Sources & References\\


In [242]:
### Functionality to search and save ADS results
### checking classification of objects in our sample

import os
import requests
import pandas as pd


### Read prior ADS search results as pandas DF
### for version control and reproducibility

ads_search = pd.read_csv("ads_search_results_df.csv")

def query_ads(snname):
    """
    Query ADS for references to new classifications of the
    objects in our sample.
    Utilizes the ADS API to search for references to published
    works on the objects in our sample
    Returns a list of published, refereed papers for each of the
    objects in our sample.
    Relies on an ADS API token as an environment variable.
    """
    token = os.environ.get("ADS_API_TOKEN", None)
    if not token:
        raise Exception("Missing ADS API Token!")
    
    url = f"https://api.adsabs.harvard.edu/v1/search/query?q=abs:{snname}&fq=property:refereed&fl=bibcode,title"

    header = {"Authorization": f"Bearer {token}"}
    
    response = requests.get(url, headers=header)
    return response

def create_new_row(name, oldtype, newtype, reference):
    """
    Create new row for ADS search result
    """
    return pd.DataFrame(
        [
            {
                "Name": name,
                "Oldtype": oldtype,
                "Newtype": newtype,
                "Reference": reference,
            }
        ]
    )

def add_row_to_df(df, row):
    return pd.concat([df, row])

def save_df(df):
    df.to_csv("ads_search_results_df.csv")

In [243]:
cited_works = {}
sn_df = filter_sne(Type="Other", Subtype="Unclassified")
for _, row in sn_df.iterrows():
    snname = row["Name"][:2] + " " + row["Name"][2:]
    response = query_ads(snname)
    if response.json()["response"]["numFound"] > 0:
        cited_works[snname] = response.json()["response"]["docs"]

In [244]:
for name, ads_result in cited_works.items():
    print(f"{name}:")
    for result in ads_result:
        print(result)

AT 2020iko:
{'bibcode': '2021AJ....161...15S', 'title': ['AT 2020iko: A WZ Sge-type Dwarf Nova Candidate with an Anomalous Precursor Event']}
AT 2023fhn:
{'bibcode': '2024A&A...691A.329C', 'title': ['Multi-wavelength observations of the luminous fast blue optical transient AT 2023fhn: Up to ∼200 days post-explosion']}
{'bibcode': '2024MNRAS.527L..47C', 'title': ['AT2023fhn (the Finch): a luminous fast blue optical transient at a large offset from its host galaxy']}
AT 2019pim:
{'bibcode': '2025MNRAS.537.2362P', 'title': ['The luminous, slow-rising orphan afterglow AT2019pim as a candidate moderately relativistic outflow']}
AT 2021aeuk:
{'bibcode': '2024ApJ...977..279B', 'title': ["Gleeok's Fire-breathing: Triple Flares of AT 2021aeuk within Five Years from the Active Galaxy SDSS J161259.83+421940.3"]}
AT 2016aps:
{'bibcode': '2021ApJ...908...99S', 'title': ['Extremely Energetic Supernova Explosions Embedded in a Massive Circumstellar Medium: The Case of SN 2016aps']}
AT 2020xnd:
{'bibc

In [270]:
# row = create_new_row(
#     name="AT2023fhn",
#     oldtype="Unclassified",
#     newtype="FBOT",
#     reference=["2024A&A...691A.329C"],
# )
# 
# ads_search = add_row_to_df(ads_search, row)

print(ads_search)

    Unnamed: 0.2  Unnamed: 0.1  Unnamed: 0        Name       Oldtype  \
0            0.0           0.0         NaN   SN2023aew         SNIIb   
1            1.0           1.0         NaN   SN2020fqv         SNIIb   
2            2.0           2.0         NaN   SN2022crv          SNIb   
3            3.0           3.0         NaN   SN2019dge          SNIb   
4            4.0           4.0         NaN   SN2021gno          SNIb   
5            5.0           5.0         NaN   SN2019ehk          SNIb   
6            6.0           6.0         NaN    SN2008bo          SNIb   
7            7.0           7.0         NaN    SN2010gx          SNIc   
8            8.0           8.0         NaN   AT2017gge  Unclassified   
9            9.0           0.0         NaN    SN2011hw         SNIIn   
10          10.0           0.0         NaN   SN2021foa         SNIIn   
11          11.0           0.0         NaN    SN2009ip         SNIIn   
12          12.0           0.0         NaN   SN2018evt          

In [271]:
# Completed: SNIIb, SNIb, SNIc, Unclassified, SNIbn, SNIcn, 
# SNIIn, SLSN-I, SLSN-II, TDE, SNII-pec, SNII-P, SNII, SNIa, 
# SNIa-91bg-like, SNIa-91T-like, SNIa-CSM, SNIa-pec, SNIax

# save_df(ads_search)

In [198]:
def get_bibtex_from_ads(bibcode):
    """
    Query ADS for references to new classifications of the
    objects in our sample.
    Utilizes the ADS API to search for references to published
    works on the objects in our sample
    Returns a list of published, refereed papers for each of the
    objects in our sample.
    Relies on an ADS API token as an environment variable.
    """
    token = os.environ.get("ADS_API_TOKEN", None)
    if not token:
        raise Exception("Missing ADS API Token!")
    
    header = {
        "Authorization": f"Bearer {token}",
        # "Content-Type": "application/json",
    }
    
    url = f"https://api.adsabs.harvard.edu/v1/export/bibtex/{bibcode}"

    response = requests.get(url, headers=header)
    return response

In [308]:
with open("sample_refs.txt", "ab+") as f:
    for ref in refs:
        r = get_bibtex_from_ads(ref)
        if r.status_code == 200:
            f.write(r.content + b'\n')
        else:
            print(f"Failed getting bib for {ref}")

Failed getting bib for 2006lcvs.book53680T


In [305]:
import pandas as pd

ads_search = pd.read_csv("ads_search_results_df.csv")

new_classification_refs = []
for row in ads_search.itertuples():
    ref_str = ''
    row_refs = row.Reference.replace("'", "").replace("[", "").replace("]", "").split(", ")
    for ref in row_refs:
        try:
            ref_ind = new_classification_refs.index(ref) + 1
        except ValueError:                            
            new_classification_refs.append(ref)
            ref_ind = len(new_classification_refs)

        ref_str += f"{ref_ind} "
    
    print(f"{row.Name} & {row.Oldtype} & {row.Newtype} & {ref_str} \\\\")

for i, ref in enumerate(new_classification_refs):
        print(f"[{i+1}]: \citet{{{ref}}}; ")


SN2023aew & SNIIb & Transitional & 1 2  \\
SN2020fqv & SNIIb & SNII & 3  \\
SN2022crv & SNIb & SNIIb & 4 5  \\
SN2021gno & SNIb & CaST & 6 7  \\
SN2019ehk & SNIb & CaST & 8 9  \\
SN2008bo & SNIb & SNIIb & 10  \\
SN2010gx & SNIc & Transitional & 11  \\
AT2017gge & Unclassified & TDE & 12  \\
SN2011hw & SNIIn & SNIbn & 13 14  \\
SN2021foa & SNIIn & Transitional & 15 16  \\
SN2009ip & SNIIn & Impostor & 17  \\
SN2018evt & SNIa & SNIa-CSM & 18  \\
SN2008ge & SNIa & SNIax & 19  \\
SN2020aeuh & SNIa & SNIa-CSM & 20  \\
SN2020eyj & SNIa & SNIa-CSM & 21  \\
SN2009dc & SNIa & SNIa-pec & 22  \\
SN2012dn & SNIa & SNIa-pec & 23  \\
SN2022esa & SNIa-CSM & Transitional & 24 25  \\
SN2012Z & SNIa-pec & SNIax & 26 27  \\
SN2005hk & SNIa-pec & SNIax & 28  \\
SN2011ay & SNIa-pec & SNIax & 29  \\
SN2008ae & SNIa-pec & SNIax & 27  \\
SN2007ax & SNIa & SNIa-91bg-like & 30  \\
SN2019yvq & SNIa & SNIa-pec & 31  \\
SN2005ke & SNIa & SNIa-91bg-like & 32  \\
SN2022joj & SNIa & SNIa-pec & 33 34  \\
SN2019hcc & S

In [302]:
with open("new_classification_refs.txt", "ab+") as f:
    for ref in new_classification_refs:
        r = get_bibtex_from_ads(ref)
        if r.status_code == 200:
            f.write(r.content + b'\n')
        else:
            print(f"Failed getting bib for {ref}")